# Explore datasets, labels, and class distributions


In [1]:
import sys; print(sys.executable)

/Users/jackteller/jev-classifier/.venv/bin/python


In [2]:
%pip install typesafe-sdk datasets pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from getpass import getpass

if not os.environ.get("TYPESAFE_API_KEY"):
    os.environ["TYPESAFE_API_KEY"] = getpass("TypeSafe API key:")

In [4]:
from typesafe_sdk import Choice, TypeSafeClient, TypeSafeAPIError

client = TypeSafeClient()

test = client.system_one(
    "I was charged twice. Please help ASAP.",
    {"tone": Choice(instructions="What is the tone?",
                    criteria={"calm": None, "angry": None})},
)
ans = test.choices["tone"]
print("choice:", ans.choice)
print("confidence:", ans.confidence)
print("probabilities:", ans.probabilities, type(ans.probabilities))

choice: angry
confidence: 0.92
probabilities: {'calm': 0.04, 'angry': 0.96} <class 'dict'>


In [7]:
import os, tarfile, urllib.request

archive = "20news-bydate.tar.gz"
if not os.path.exists("20news-bydate-train"):
    if not os.path.exists(archive):
        urllib.request.urlretrieve(
            "http://qwone.com/~jason/20Newsgroups/20news-bydate.tar.gz", archive)
    with tarfile.open(archive, "r:gz") as tar:
        tar.extractall()

print(os.getcwd())
print([d for d in os.listdir(".") if d.startswith("20news")])

/Users/jackteller/jev-classifier/notebooks
['20news-bydate.tar.gz', '20news-bydate-test', '20news-bydate-train']


In [8]:
import pandas as pd
from datasets import load_dataset

# 20 Newsgroups
def load_newsgroups(main_dir):
    listdir = sorted(d for d in os.listdir(main_dir)
                     if os.path.isdir(os.path.join(main_dir, d)))
    datalist = []
    for dirname in listdir:
        listfiles = sorted(os.listdir(os.path.join(main_dir, dirname)))
        for filename in listfiles:
            with open(os.path.join(main_dir, dirname, filename),
                      encoding="latin1", errors="ignore") as f:
                text = f.read()
            datalist.append((dirname, filename, text))
    df = pd.DataFrame(datalist, columns=["label", "filename", "text"])
    return df[["text", "label"]]

# Hugging Face datasets
def load_hf(names, text_col="text"):
    last_err = None
    for name in names:               
        try:
            ds = load_dataset(name)
            break
        except Exception as e:
            last_err = e
    else:
        raise last_err
    label_names = ds["train"].features["label"].names
    out = {}
    for split in ["train", "test"]:
        df = ds[split].to_pandas()
        out[split] = pd.DataFrame({"text": df[text_col],
                                   "label": [label_names[i] for i in df["label"]]})
    return out

data = {
    "20newsgroups": {"train": load_newsgroups("20news-bydate-train"),
                     "test":  load_newsgroups("20news-bydate-test")},
    "ag_news":      load_hf(["fancyzhx/ag_news", "ag_news"]),
    "banking77":    load_hf(["PolyAI/banking77", "legacy-datasets/banking77"]),
}

Generating test split: 100%|██████████| 3080/3080 [00:00<00:00, 520968.52 examples/s]


In [9]:
for name, splits in data.items():
    print(f"{name:14s} train={len(splits['train']):>7,}  test={len(splits['test']):>6,}  "
          f"classes={splits['train']['label'].nunique()}")

20newsgroups   train= 11,314  test= 7,532  classes=20
ag_news        train=120,000  test= 7,600  classes=4
banking77      train= 10,003  test= 3,080  classes=77


In [11]:
import time

def jev_classify(text, labels, instructions="Which category does this text belong to?",
                 max_chars=8000):
    """Return (predicted label, {label: prob}, latency seconds, usage)."""
    t0 = time.perf_counter()
    resp = client.system_one(
        text[:max_chars],
        {"label": Choice(instructions=instructions,
                         criteria={lab: None for lab in labels})},
    )
    latency = time.perf_counter() - t0
    ans = resp.choices["label"]
    probs = dict(ans.probabilities)   
    return ans.choice, probs, latency, getattr(resp, "usage", None)

for name, splits in data.items():
    labels = sorted(splits["train"]["label"].unique())
    row = splits["test"].iloc[0]
    pred, probs, lat, usage = jev_classify(row["text"], labels)
    top3 = sorted(probs.items(), key=lambda kv: -kv[1])[:3]
    print(f"{name}: true={row['label']} | pred={pred} | {lat:.2f}s | usage={usage}")
    print("   top 3:", [(k, round(v, 3)) for k, v in top3])

20newsgroups: true=alt.atheism | pred=talk.religion.misc | 0.18s | usage=input_tokens=754 output_tokens=237
   top 3: [('talk.religion.misc', 0.52), ('soc.religion.christian', 0.41), ('alt.atheism', 0.07)]
ag_news: true=Business | pred=Business | 0.08s | usage=input_tokens=341 output_tokens=47
   top 3: [('Business', 0.99), ('World', 0.01), ('Sci/Tech', 0.0)]
banking77: true=card_arrival | pred=lost_or_stolen_card | 0.08s | usage=input_tokens=1016 output_tokens=829
   top 3: [('lost_or_stolen_card', 0.94), ('card_arrival', 0.04), ('card_delivery_estimate', 0.02)]


In [12]:
import json, csv

os.makedirs("results", exist_ok=True)

def run_jev(name, n=None, max_chars=8000):
    test = data[name]["test"]
    if n:
        test = test.sample(n=n, random_state=42)   # quick trial run
    labels = sorted(data[name]["train"]["label"].unique())
    out_path = f"results/jev_{name}{'_n'+str(n) if n else ''}.csv"

    done = set()
    if os.path.exists(out_path):
        done = set(pd.read_csv(out_path)["idx"])
        print(f"Resuming: {len(done)} already done")

    new_file = not os.path.exists(out_path)
    with open(out_path, "a", newline="") as f:
        writer = csv.writer(f)
        if new_file:
            writer.writerow(["idx", "true", "pred", "max_prob", "probs_json",
                             "latency_s", "input_tokens", "output_tokens", "error"])
        for i, (idx, row) in enumerate(test.iterrows()):
            if idx in done:
                continue
            try:
                pred, probs, lat, usage = jev_classify(row["text"], labels,
                                                       max_chars=max_chars)
                writer.writerow([idx, row["label"], pred, max(probs.values()),
                                 json.dumps(probs), round(lat, 4),
                                 getattr(usage, "input_tokens", None),
                                 getattr(usage, "output_tokens", None), ""])
            except Exception as e:
                writer.writerow([idx, row["label"], "", "", "", "", "", "", repr(e)])
            f.flush()
            if i % 250 == 0:
                print(f"{name}: {i}/{len(test)}")
    print(f"Saved -> {out_path}")
    return pd.read_csv(out_path)

In [13]:
for name in data:
    df = run_jev(name, n=200)
    ok = df[df["error"].isna()]
    print(f"{name}: acc={(ok['true'] == ok['pred']).mean():.3f}  "
          f"errors={df['error'].notna().sum()}  "
          f"p95 latency={ok['latency_s'].quantile(0.95):.3f}s\n")

20newsgroups: 0/200
Saved -> results/jev_20newsgroups_n200.csv
20newsgroups: acc=0.800  errors=0  p95 latency=0.125s

ag_news: 0/200
Saved -> results/jev_ag_news_n200.csv
ag_news: acc=0.885  errors=0  p95 latency=0.165s

banking77: 0/200
Saved -> results/jev_banking77_n200.csv
banking77: acc=0.755  errors=0  p95 latency=0.166s



In [14]:
for name in ["ag_news", "banking77", "20newsgroups"]:
    run_jev(name)

ag_news: 0/7600
ag_news: 250/7600
ag_news: 500/7600
ag_news: 750/7600
ag_news: 1000/7600
ag_news: 1250/7600
ag_news: 1500/7600
ag_news: 1750/7600
ag_news: 2000/7600
ag_news: 2250/7600
ag_news: 2500/7600
ag_news: 2750/7600
ag_news: 3000/7600
ag_news: 3250/7600
ag_news: 3500/7600
ag_news: 3750/7600
ag_news: 4000/7600
ag_news: 4250/7600
ag_news: 4500/7600
ag_news: 4750/7600
ag_news: 5000/7600
ag_news: 5250/7600
ag_news: 5500/7600
ag_news: 5750/7600
ag_news: 6000/7600
ag_news: 6250/7600
ag_news: 6500/7600
ag_news: 6750/7600
ag_news: 7000/7600
ag_news: 7250/7600
ag_news: 7500/7600
Saved -> results/jev_ag_news.csv
banking77: 0/3080
banking77: 250/3080
banking77: 500/3080
banking77: 750/3080
banking77: 1000/3080
banking77: 1250/3080
banking77: 1500/3080
banking77: 1750/3080
banking77: 2000/3080
banking77: 2250/3080
banking77: 2500/3080
banking77: 2750/3080
banking77: 3000/3080
Saved -> results/jev_banking77.csv
20newsgroups: 0/7532
20newsgroups: 250/7532
20newsgroups: 500/7532
20newsgroups: 7